In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, HBox, VBox, Layout

t = np.linspace(-5,5,500)
dt = t[1]-t[0]

def rect_pulse(t, width, height, center):
    return height*((t >= center - width/2) & (t <= center + width/2)).astype(float)

def convolution_demo(A, alpha, B, beta, time_frame):
    x = rect_pulse(t, width=alpha, height=A, center=0)
    
    start_center = -5 - beta/2
    shift = time_frame*(10/len(t))
    y_center = start_center + shift
    y = rect_pulse(t, width=beta, height=B, center=y_center)
    
    overlap = np.minimum(x, y)
    
    y_reversed = y[::-1]
    z = np.zeros_like(t)
    shift_samples = int(time_frame)
    for i in range(shift_samples+1):
        if i < len(t):
            temp = np.zeros_like(t)
            temp[i:] = y_reversed[:len(t)-i]
            z[i] = np.trapz(x * temp, t)
    
    x_ref = rect_pulse(t, width=alpha, height=A, center=0)
    y_ref = rect_pulse(t, width=beta, height=B, center=0)
    
    fig, axs = plt.subplots(2,2,figsize=(10,5))
    plt.subplots_adjust(hspace=0.4)
    
    # Πάνω αριστερά: x reference
    axs[0,0].plot(t,x_ref,'b',lw=2)
    axs[0,0].set_title("Signal x(τ) (reference)")
    axs[0,0].set_ylim(0, max(A,B)+0.5)
    axs[0,0].grid(True)
    
    # Πάνω δεξιά: y reference
    axs[0,1].plot(t,y_ref,'g',lw=2)
    axs[0,1].set_title("Signal y(τ) (reference)")
    axs[0,1].set_ylim(0, max(A,B)+0.5)
    axs[0,1].grid(True)
    
    # Κάτω αριστερά: sliding y
    axs[1,0].plot(t,x,'b',lw=2)        # x σταθερός μπλε
    axs[1,0].plot(t,y,'g',lw=2)        # y sliding πράσινο, συνεχές
    axs[1,0].fill_between(t,0,overlap,color='orange',alpha=0.5)
    axs[1,0].set_title(f"Sliding y(-τ) at frame {time_frame}")
    axs[1,0].set_ylim(0, max(A,B)+0.5)
    axs[1,0].grid(True)
    
    # Κάτω δεξιά: convolution
    axs[1,1].plot(t,z,'m',lw=2)
    axs[1,1].fill_between(t,0,z,color='orange',alpha=0.5)
    axs[1,1].set_title("Convolution z(t)")
    axs[1,1].set_ylim(0, max(A,B)+0.5)
    axs[1,1].grid(True)
    
    plt.show()

# Sliders
A_slider = FloatSlider(value=1, min=0.5, max=3, step=0.1, description='A', layout=Layout(width='45%'))
alpha_slider = FloatSlider(value=2, min=0.5, max=4, step=0.1, description='α', layout=Layout(width='45%'))
B_slider = FloatSlider(value=1, min=0.5, max=3, step=0.1, description='B', layout=Layout(width='45%'))
beta_slider = FloatSlider(value=2, min=0.5, max=4, step=0.1, description='β', layout=Layout(width='45%'))
time_slider = FloatSlider(value=0, min=0, max=len(t)-1, step=1, description='Time', layout=Layout(width='90%'))

slider_box1 = HBox([A_slider, alpha_slider])
slider_box2 = HBox([B_slider, beta_slider])
all_sliders = VBox([slider_box1, slider_box2, time_slider])

interact(convolution_demo,
         A=A_slider,
         alpha=alpha_slider,
         B=B_slider,
         beta=beta_slider,
         time_frame=time_slider)